In [5]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 300

superficie = rng.normal(90, 25, size=n).clip(30, None)          # m^2
habitaciones = rng.integers(1, 6, size=n).astype(float)
ciudad = rng.choice(["Xalapa", "CDMX", "Monterrey"], size=n,
                     p=[0.5, 0.3, 0.2])
antiguedad = rng.integers(0, 40, size=n).astype(float)           # años

# Precio "verdadero" simulado: relacion lineal + ruido aleatorio
bonus_ciudad = pd.Series(ciudad).map(
    {"Xalapa": 0, "CDMX": 450, "Monterrey": 200}).to_numpy()

precio = (
    15 * superficie
    + 80 * habitaciones
    - 3 * antiguedad
    + bonus_ciudad
    + rng.normal(0, 150, size=n)
)

df = pd.DataFrame({
    "superficie": superficie,
    "habitaciones": habitaciones,
    "ciudad": ciudad,
    "antiguedad": antiguedad,
    "precio": precio,
})

# Introducimos valores faltantes A PROPOSITO8
faltantes_idx = rng.choice(df.index, size=20, replace=False)
df.loc[faltantes_idx, "habitaciones"] = np.nan

df.head()

,superficie,habitaciones,ciudad,antiguedad,precio
0,97.617927,NaN,Monterrey,24.0,1946.102881
1,64.000397,2.0,Xalapa,1.0,1259.153402
2,108.761280,NaN,CDMX,12.0,2038.753463
3,113.514118,2.0,CDMX,5.0,2121.916030
4,41.224120,3.0,CDMX,0.0,1144.363226


In [7]:
# TODO: usa df.info() para ver tipos de datos y cuántos valores
#       no nulos hay en cada columna (fíjate en 'habitaciones').

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   superficie    300 non-null    float64
 1   habitaciones  280 non-null    float64
 2   ciudad        300 non-null    object 
 3   antiguedad    300 non-null    float64
 4   precio        300 non-null    float64
dtypes: float64(4), object(1)
memory usage: 11.8+ KB


Se puede observar que la columna de habitaciones tiene 20 datos faltantes

In [8]:
# TODO: usa df.describe() para ver estadísticos básicos
#       de las columnas numéricas (media, desviación, cuartiles).

df.describe()

,superficie,habitaciones,antiguedad,precio
count,300.000000,280.000000,300.000000,300.000000
mean,88.986916,3.032143,18.956667,1692.580762
std,23.220181,1.455075,11.776974,445.229373
min,30.000000,1.000000,0.000000,590.322203
25%,73.265883,2.000000,9.000000,1392.680281
50%,87.825240,3.000000,19.000000,1703.704045
75%,101.842244,4.000000,29.000000,1966.741023
max,162.846562,5.000000,39.000000,3319.653720


Las habitaciones son las que tienen menos datos por lo que se puede decir que tiene datos faltantes y ademas de eso el precio es la columna con mayor varianza

In [14]:
# TODO: usa df["ciudad"].value_counts() para ver cuántas
#       viviendas hay de cada ciudad.
df["ciudad"].value_counts()

ciudad
Xalapa       155
CDMX          89
Monterrey     56
Name: count, dtype: int64

Xalapa es la ciudad con mas viviendas en venta

## 1.5 Particion de datos

In [16]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["precio"])
y = df["precio"]

In [19]:
#| label: particion-todo
#| eval: false

# TODO: separa X, y en X_train, X_test, y_train, y_test.
#       Usa test_size=0.2 y random_state=0.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
X_train.shape,X_test.shape

((240, 4), (60, 4))

## 1.6 Preprocesamiento de los datos

In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

columnas_numericas = ["superficie", "habitaciones", "antiguedad"]
columnas_categoricas = ["ciudad"]

In [22]:
# Pipeline para columnas numéricas: imputar mediana + escalar
pipeline_numerico = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    # TODO: agrega un paso llamado "escalar" con StandardScaler()
    ("scaler",StandardScaler())
])

# Pipeline para columnas categóricas: codificación one-hot
pipeline_categorico = Pipeline([
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

# TODO: combina ambos pipelines en un ColumnTransformer.
#       Pista: recibe una lista de tuplas (nombre, pipeline, columnas)
preprocesador = ColumnTransformer(transformers=[
     ("num", pipeline_numerico, columnas_numericas),
     ("cat", pipeline_categorico, columnas_categoricas),
])

## 1.7

In [26]:
from sklearn.linear_model import Ridge

modelo = Pipeline([
    ("preprocesamiento", preprocesador), ("regresor", Ridge(alpha=1.0))
    # TODO: agrega el regresor, con el nombre "regresor",
    #       por ejemplo Ridge(alpha=1.0)
])

# TODO: entrena el modelo con modelo.fit(X_train, y_train)
modelo.fit(X_train,y_train)

Pipeline(steps=[('preprocesamiento',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputar',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['superficie', 'habitaciones',
                                                   'antiguedad']),
                                                 ('cat',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['ciudad'])])),
                ('regresor', Ridge())])

## 1.8 Evaluación del modelo

In [33]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: obtén las predicciones sobre X_test
y_pred = modelo.predict(X_test)

# TODO: calcula RMSE, MAE y R^2
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
MAE = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test,y_pred)

In [34]:
print(RMSE)
print(MAE)
print(r2)

145.452620916545
111.47597440198582
0.9017987448744508
